# Verifier-Guided Reasoning MVP

This notebook is the public Colab orchestration surface for the project. It keeps the repository's legacy name, but the actual implementation is positioned as a verifier-guided process supervision system rather than a full RLHF stack.

Notebook goals:

- install the repo and Colab extras,
- mount Google Drive,
- configure DVC and MLflow paths on Drive,
- run the arithmetic verifier demo,
- preview the SFT path for `Qwen/Qwen2.5-1.5B-Instruct`.


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print({'in_colab': IN_COLAB, 'python': sys.version.split()[0]})

if IN_COLAB:
    get_ipython().run_line_magic('pip', 'install -q -e .[colab,dev]')


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DVC_REMOTE_PATH = Path('/content/drive/MyDrive/rlhf_logic_verification/dvc')
MLFLOW_ROOT = Path('/content/drive/MyDrive/rlhf_logic_verification/mlruns')
DVC_REMOTE_PATH.mkdir(parents=True, exist_ok=True)
MLFLOW_ROOT.mkdir(parents=True, exist_ok=True)
print({'dvc_remote': str(DVC_REMOTE_PATH), 'mlflow_root': str(MLFLOW_ROOT)})


In [ ]:
if IN_COLAB:
    get_ipython().system('dvc init || true')
    get_ipython().system(f'dvc remote add -d colab_drive {DVC_REMOTE_PATH} || true')

os.environ['MLFLOW_TRACKING_URI'] = MLFLOW_ROOT.resolve().as_uri()
print('MLFLOW_TRACKING_URI=', os.environ['MLFLOW_TRACKING_URI'])


In [ ]:
from verifier_guided_reasoning.pipeline import run_small_demo

summary = run_small_demo(
    output_path='artifacts/eval/demo_summary.json',
    report_markdown_path='artifacts/eval/demo_report.md',
    tracker_root=str(MLFLOW_ROOT) if IN_COLAB else 'mlruns',
)
summary


In [ ]:
from pathlib import Path

report_path = Path('artifacts/eval/demo_report.md')
print(report_path.read_text(encoding='utf-8'))


## Arithmetic SFT Path

The public MVP stays arithmetic-first:

- `MU-NLPC/Calc-svamp` for seed structured traces
- `openai/gsm8k` for benchmark and augmentation
- verifier gates before SFT export
- `Qwen/Qwen2.5-1.5B-Instruct` as the default base model
- `Qwen/Qwen2.5-Math-1.5B-Instruct` as the optional comparison model

Do not move to DPO until the verifier and accepted-trace pipeline are stable.


In [ ]:
TRAINING_CONFIG = {
    'base_model': 'Qwen/Qwen2.5-1.5B-Instruct',
    'comparison_model': 'Qwen/Qwen2.5-Math-1.5B-Instruct',
    'tuning_method': 'qlora',
    'max_seq_length': 2048,
    'best_of_n': 4,
    'datasets': {
        'seed': 'MU-NLPC/Calc-svamp',
        'benchmark': 'openai/gsm8k',
        'logic_eval': 'tasksource/folio',
    },
}
TRAINING_CONFIG


In [ ]:
# Optional next step once you are ready for dataset-backed runs:
# !python scripts/prepare_datasets.py --demo --output data/processed/demo_arithmetic.jsonl
# !python scripts/run_small_demo.py --input data/processed/demo_arithmetic.jsonl \
#     --output artifacts/eval/demo_summary.json \
#     --report-md artifacts/eval/demo_report.md \
#     --tracker-root "$MLFLOW_TRACKING_URI"

print('Notebook scaffold complete. Move into real SFT only after accepted traces are stable.')
